In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.optimize import brentq
from scipy.stats import norm

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from triplet_lineage.theory import compute_delta_star, sample_complexity_bound

CAPACITY_CSV = PROJECT_ROOT / "results" / "fig1_required_capacity.csv"
RELIABILITY_CSV = PROJECT_ROOT / "results" / "fig2_success_probability.csv"
FIGURE_PATH = PROJECT_ROOT / "figures" / "figure-04.pdf"

for input_path in (CAPACITY_CSV, RELIABILITY_CSV):
    if not input_path.exists():
        raise FileNotFoundError(f"Missing required input file: {input_path}")

capacity_df = pd.read_csv(CAPACITY_CSV)
reliability_df = pd.read_csv(RELIABILITY_CSV)


def infer_triplet_error(target_accuracy, n_triplets, success_probability=0.9):
    if not 0 < target_accuracy < 1:
        return np.nan

    z_score = norm.ppf(success_probability)
    upper = 1.0 - target_accuracy - 1e-12
    if upper <= 1e-12:
        return np.nan

    def objective(error_rate):
        variance = error_rate * (1.0 - error_rate) * n_triplets
        if variance <= 0:
            return np.inf
        numerator = (1.0 - error_rate - target_accuracy) * n_triplets
        return numerator / np.sqrt(variance) - z_score

    try:
        return brentq(objective, 1e-12, upper)
    except ValueError:
        return np.nan


def theoretical_site_count(
    target_accuracy,
    depth=6,
    p_miss=0.0,
    lam=0.5,
    n_states=5,
    success_probability=0.9,
):
    n_cells = 2 ** depth
    n_triplets = n_cells * (n_cells - 1) * (n_cells - 2) / 6
    q_collision = 1.0 / n_states
    l_star = 1.0 / depth
    delta_star = compute_delta_star(lam, q_collision, d_max=1.0)
    triplet_error = infer_triplet_error(target_accuracy, n_triplets, success_probability)

    if not np.isfinite(triplet_error) or triplet_error <= 0:
        return np.nan

    return sample_complexity_bound(
        error=triplet_error,
        q=q_collision,
        l_star=l_star,
        lam=lam,
        delta_star=delta_star,
        p_miss=p_miss,
    )


In [ ]:
sns.set_theme(
    style="whitegrid",
    rc={
        "axes.edgecolor": "black",
        "grid.linestyle": "--",
        "grid.color": "lightgrey",
        "grid.alpha": 0.7,
        "lines.solid_capstyle": "round",
        "lines.solid_joinstyle": "round",
    },
)

capacity_clean = capacity_df.loc[capacity_df["Required_k"] > 0].copy()

accuracy_grid = np.linspace(0.01, 0.999, 500)
theory_rows = [
    {
        "Required_k": theoretical_site_count(acc, p_miss=0.0),
        "Target_Accuracy": acc,
    }
    for acc in accuracy_grid
]
theory_df = pd.DataFrame(theory_rows).replace([np.inf, -np.inf], np.nan).dropna()

display_x_max = max(float(capacity_clean["Required_k"].max()), float(theory_df["Required_k"].max()))
capacity_anchor = pd.DataFrame({"Required_k": [0.0], "Target_Accuracy": [0.0]})
capacity_endpoint = pd.DataFrame({"Required_k": [display_x_max], "Target_Accuracy": [1.0]})
capacity_plot = pd.concat([capacity_anchor, capacity_clean, capacity_endpoint], ignore_index=True)
theory_df = pd.concat(
    [
        pd.DataFrame({"Required_k": [0.0], "Target_Accuracy": [0.0]}),
        theory_df,
        pd.DataFrame({"Required_k": [display_x_max], "Target_Accuracy": [0.999]}),
    ],
    ignore_index=True,
).sort_values("Required_k")

fig, (ax_capacity, ax_reliability) = plt.subplots(1, 2, figsize=(12, 4.5))

sns.lineplot(
    data=capacity_plot,
    x="Required_k",
    y="Target_Accuracy",
    color="#FF8C00",
    linewidth=2.5,
    marker="o",
    markersize=6,
    markeredgecolor="white",
    errorbar=("ci", 95),
    err_kws={"alpha": 0.25},
    ax=ax_capacity,
    label="Simulated MAX-Cut (mean with 95% CI)",
)

ax_capacity.plot(
    theory_df["Required_k"],
    theory_df["Target_Accuracy"],
    color="#4682B4",
    linestyle="--",
    linewidth=2.5,
    label="Theoretical bound",
)
ax_capacity.axhline(1.0, color="red", linestyle=":", linewidth=2, alpha=0.8, label="Ideal triplet oracle")

ax_capacity.set_xscale("symlog", linthresh=10)
ax_capacity.set_xlim(left=0, right=display_x_max * 1.05)
ax_capacity.set_xticks([0, 10, 100, 1000, 10000, 20000])
ax_capacity.set_xlabel("Required sites (n_site)")
ax_capacity.set_ylabel("Target accuracy")
ax_capacity.set_ylim(-0.05, 1.05)
ax_capacity.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}" if x >= 1 else f"{x:g}"))
ax_capacity.legend(frameon=False, fontsize=9)

sns.lineplot(
    data=reliability_df,
    x="k_cand",
    y="Success_Probability",
    hue="p_miss",
    style="p_miss",
    markers=True,
    dashes=False,
    linewidth=2.5,
    markersize=7,
    markeredgecolor="white",
    markeredgewidth=1.2,
    errorbar=("ci", 95),
    ax=ax_reliability,
)
ax_reliability.axhline(y=0.9, color="red", linestyle="--", linewidth=1.5, label="90% reliability")
ax_reliability.set_xlabel("Number of sites (n_site)")
ax_reliability.set_ylabel("Success probability")
ax_reliability.set_ylim(-0.05, 1.05)
ax_reliability.legend(title="Missing rate", loc="lower right", frameon=False, fontsize=9)

for axis in (ax_capacity, ax_reliability):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["bottom"].set_linewidth(1.2)
    axis.spines["left"].set_linewidth(1.2)

fig.tight_layout()
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved Figure 4 to {FIGURE_PATH}")
